# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process a rich scientific dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, which is designed for FAIR and reproducible machine learning data pipelines.

### Dataset Source
The dataset is described and packaged using the [ML Commons Croissant schema](https://mlcommons.org/croissant/) and is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load and introspect the dataset metadata and records using the Croissant API.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant metadata URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Let's list the available record sets (`@id`), their fields, and important details. Every entity is referenced by its `@id` as per the Croissant schema best practices.

In [ ]:
# List all record sets and their fields with @id
print("Available record sets and fields:")
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"\nRecord set @id: {rs.id}")
    print(f"  name: {rs.name}")
    print(f"  description: {rs.description}")
    print(f"  fields:")
    for f in rs.fields:
        print(f"    - field @id: {f.id}, name: {f.name}, dataType: {f.data_type}")

## 3. Data Extraction
We'll choose a record set that contains the main protein quantification table and load it into a DataFrame for further exploration.

**Action:**
- Identify the main data record set using its `@id` (see above output), for example, `cr:MassSpecProteinTable`.
- Reference fields/columns using their `@id` as well.
- We'll extract all available record sets for demonstration.

In [ ]:
# Gather all record set @id's
record_sets_ids = [rs.id for rs in metadata.record_sets]
print("Record set @id's in this dataset:")
pprint.pprint(record_sets_ids)

# Load all record sets into DataFrames
dataframes = {}
for record_set_id in record_sets_ids:
    print(f"Loading records from record set {record_set_id}...")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"{record_set_id}: loaded {len(records)} records with columns: {dataframes[record_set_id].columns.tolist()}")
        else:
            print(f"{record_set_id}: No records in this record set.")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")

### Example: Inspect the Main Protein Table
Let's select the protein table for further exploration. Please replace `cr:MassSpecProteinTable` with the actual main record set `@id` if different.

In [ ]:
# Example: use the @id for the main protein table.
main_table_id = next((r.id for r in metadata.record_sets if 'protein' in r.name.lower()), record_sets_ids[0])
df = dataframes[main_table_id]
print(f"Using record set: {main_table_id}")
print(f"Columns: {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)

Let's process a numeric field, for example, the normalized abundance or peptide count values. We'll filter, normalize, and group as demonstration.

We'll use:
- `cr:peptideCount`: total number of peptides (numeric field)
- `cr:sample`: a group/categorical field if available

In [ ]:
# Let us discover a numeric field to analyze
numeric_field = None
group_field = None
# Find a numeric field (e.g., peptide count, MW, Abundance)
for col in df.columns:
    if 'peptide' in col.lower() or 'abundance' in col.lower() or 'mw' in col.lower():
        numeric_field = col
        break

# Find a potential grouping field
for col in df.columns:
    if 'sample' in col.lower() or 'group' in col.lower():
        group_field = col
        break

print(f"Numeric field: {numeric_field}, Group field: {group_field}")

if numeric_field is not None:
    threshold = df[numeric_field].median() if df[numeric_field].dtype != object else 1 # For demo
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nMean {numeric_field} grouped by {group_field}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found in the main table.")

## 5. Visualization

Let's visualize the distribution of the main numeric field (e.g., peptide count, normalized abundance) and how it breaks down by group/sample if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
if numeric_field:
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

if group_field and group_field in df.columns and numeric_field:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

This notebook demonstrated the use of the `mlcroissant` library to access and process a mass spectrometry dataset described with a Croissant schema. 

**Key points:**
- All data access and manipulation referenced entities by their `@id`, ensuring reproducibility.
- The notebook showed how to load, list, extract, process, and visualize high-value protein data from human mast cell extracellular vesicles.
- This Croissant-based pipeline can be adapted for any FAIR-compliant scientific dataset.